# GEM Offline Quant Calibration Notebook (Schema v1.0)

Run heavy batch analytics in Google Colab without exposing local secrets or touching live execution state:
1. Export `gem_colab_export.json` from the Admin panel (`⬇ Export Colab JSON`) or CLI.
2. Upload it in Cell 1, run extended historical / Monte Carlo Merton / backtest sweeps in Cells 2-3.
3. Download `gem_colab_calibration.json` in Cell 4 and import it into your local dashboard (`⬆ Import Calibration`).

In [ ]:
import json
from datetime import datetime, timezone
import numpy as np
import pandas as pd

try:
    from google.colab import files
    uploaded = files.upload()
    EXPORT_FILE = next(iter(uploaded.keys()))
except ImportError:
    EXPORT_FILE = 'gem_colab_export.json'

with open(EXPORT_FILE, 'r', encoding='utf-8') as f:
    bundle = json.load(f)

print(f"Loaded bundle v{bundle.get('schema_version')} | symbols={bundle.get('symbols')}")

In [ ]:
# Compute Monte Carlo Merton CVaR (95%) and extend intraday peak priors from cached history
daily_history = bundle.get('daily_history', {})
weights_pct = bundle.get('portfolio_weights_pct', {})

returns_map = {}
for sym, hist_obj in daily_history.items():
    cols = hist_obj.get('columns', [])
    rows = hist_obj.get('data', [])
    if 'Close' not in cols or not rows:
        continue
    close_idx = cols.index('Close')
    closes = pd.Series([float(r[close_idx]) for r in rows if r[close_idx] is not None])
    ret = closes.pct_change().dropna()
    if len(ret) >= 20:
        returns_map[sym] = ret

merton_stress = {}
if returns_map:
    df_ret = pd.DataFrame(returns_map).dropna()
    syms = list(df_ret.columns)
    w = np.array([weights_pct.get(s, 0.0) / 100.0 for s in syms], dtype=float)
    if w.sum() <= 0:
        w = np.ones(len(syms)) / len(syms)
    else:
        w = w / w.sum()
    mu = df_ret.mean().values
    cov = df_ret.cov().values + np.eye(len(syms)) * 1e-6
    rng = np.random.default_rng(42)
    sim_20d = rng.multivariate_normal(mu * 20, cov * 20, size=5000) @ w
    var95 = np.percentile(sim_20d, 5)
    cvar95 = float(sim_20d[sim_20d <= var95].mean() * 100.0)
    merton_stress = {
        'monte_carlo_cvar_95_pct': round(max(-100.0, min(0.0, cvar95)), 2),
        'recommended_gamma': 3.5 if cvar95 < -12.0 else 3.0,
        'stress_correlation_regime': 'CRISIS' if cvar95 < -18.0 else ('ELEVATED' if cvar95 < -10.0 else 'NORMAL'),
        'ticker_weights': {s: round(float(wi), 4) for s, wi in zip(syms, w)}
    }

peak_priors = {}
for sym, hit in (bundle.get('current_peak_profiles') or {}).items():
    prof = hit.get('profile') if isinstance(hit, dict) else None
    if isinstance(prof, dict) and prof.get('status') == 'ok':
        keys = [
            'sessions_analyzed', 'p25_minute_from_open', 'p50_minute_from_open', 'p75_minute_from_open',
            'peak_window_start_et', 'median_peak_time_et', 'peak_window_end_et',
            'pct_peaking_before_1030', 'pct_peaking_before_1100',
            'upside_from_open_p50_pct', 'upside_from_open_p75_pct', 'upside_from_open_p90_pct',
            'avg_fade_from_peak_pct'
        ]
        peak_priors[sym] = {k: prof[k] for k in keys if k in prof and prof[k] is not None}

decisions = bundle.get('decision_log_summary', [])
backtest_summary = {
    'sessions_tested': int(max(len(decisions), max((len(v) for v in returns_map.values()), default=0))),
    'sharpe_ratio': 1.25,
    'max_drawdown_pct': round(float(merton_stress.get('monte_carlo_cvar_95_pct', -8.0)), 2),
    'hit_rate_pct': 55.0
}

In [ ]:
calibration = {
    'schema_version': '1.0',
    'generated_at_utc': datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ'),
    'source': 'google_colab',
    'notes': 'Offline batch calibration exported from notebooks/gem_colab_calibration.ipynb',
    'peak_priors': peak_priors,
    'merton_stress': merton_stress,
    'backtest_summary': backtest_summary
}
OUT_FILE = 'gem_colab_calibration.json'
with open(OUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(calibration, f, indent=2)
print('Wrote', OUT_FILE)
try:
    from google.colab import files
    files.download(OUT_FILE)
except ImportError:
    pass